# TP2 — Submit to the Weather Prediction Competition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session3/tp2.ipynb)

## Objective

Take the 3 models you trained in TP1, wrap them into an `Agent` class, test it locally (reproducing the platform's evaluation), and submit to the live competition.

### Competition Info

| | |
|---|---|
| **Enroll** | [ml-arena.com/enroll/6db4734d376f7b64249f503f140db96f](https://ml-arena.com/enroll/6db4734d376f7b64249f503f140db96f) |
| **Competition page** | [ml-arena.com/viewcompetition/26](https://ml-arena.com/viewcompetition/26) |
| **Leaderboard assessed** | 30-day rolling average — last submission April 10 / assessed May 15, 2026 |
| **Presentation** | ~April 10, 2026 |
| **Submission** | Web interface — upload `agent.py` + `model_temperature.pkl` + `model_wind_speed.pkl` + `model_rain.pkl` |

### Roadmap

| Section | What you do |
|---------|-------------|
| 1. Enroll & Understand | Join the competition, understand input/output |
| 2. Build the Agent | Bridge TP1 features → Agent class |
| 3. Test Locally | Reproduce platform evaluation with real data |
| 4. Submit | Upload to the competition |
| 5. Ideas for Improvement | Guidance for the next month |

---
## 1. Setup

In [1]:
!pip install scikit-learn pandas numpy joblib


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path


---
## 2. Enroll & Understand the Competition

### Exercise 1.1 — Enroll

1. Go to the enrollment link: [ml-arena.com/enroll/6db4734d376f7b64249f503f140db96f](https://ml-arena.com/enroll/6db4734d376f7b64249f503f140db96f)
2. Create an account (or log in)
3. Explore the competition page: leaderboard, description, rules

### Exercise 1.2 — Understand the input/output

Your agent receives a 3D numpy array `X_test` of shape `(20, 24, 8)` and must return a prediction of shape `(3,)`.

**Input: `X_test` — shape `(20, 24, 8)`**

| Axis | Size | Meaning |
|------|------|---------|
| 0 | 20 | Cities (alphabetical order) |
| 1 | 24 | Hours of history (oldest → newest) |
| 2 | 8 | Weather features |

**20 cities** (alphabetical — axis 0):

| Index | City | Index | City |
|-------|------|-------|------|
| 0 | Amsterdam | 10 | Köln |
| 1 | Barcelona | 11 | London |
| 2 | Birmingham | 12 | Manchester |
| 3 | Brussels | 13 | Marseille |
| 4 | Copenhagen | 14 | Milan |
| 5 | Dortmund | 15 | Munich |
| 6 | Dublin | 16 | **Paris** |
| 7 | Düsseldorf | 17 | Rotterdam |
| 8 | Essen | 18 | Stuttgart |
| 9 | Frankfurt am Main | 19 | Turin |

**8 features** (axis 2):

| Index | Feature | Unit |
|-------|---------|------|
| 0 | temperature | °C |
| 1 | rain | mm |
| 2 | wind_speed | m/s |
| 3 | wind_direction | degrees |
| 4 | humidity | % |
| 5 | clouds | % |
| 6 | visibility | m |
| 7 | snow | mm |

**Output: prediction — shape `(3,)`**

| Index | Target | Unit |
|-------|--------|------|
| 0 | temperature | °C |
| 1 | wind_speed | m/s |
| 2 | rain | mm |

**Question:** Where is Paris in the input array? What is `X_test[16, -1, 0]`?

`X_test.shape == (20, 24, 8)` means:

- **20** cities
- **24** past hourly observations
- **8** weather features per city and hour

The agent must transform this 3D history window into the same flat feature vector used in TP1, then return:

`[temperature_T+6h_for_Paris, wind_speed_T+6h_for_Paris, rain_T+6h_for_Paris]`


### Exercise 1.3 — Understand the scoring metric

The competition uses **Negated Normalized MAE**:

```
score = -mean(|pred - true| / std)
```

where:
- `std_temperature = 7.49 °C`
- `std_wind_speed = 5.05 m/s`
- `std_rain = 0.40 mm`

**Question:** Why is rain the hardest to predict well? (Think about what happens when you're off by 1mm on rain vs 1°C on temperature.)

Rain is the hardest target because the score is **normalized by a very small standard deviation**:

- temperature std = **7.49**
- wind_speed std = **5.05**
- rain std = **0.40**

So an error of **1 mm** on rain becomes `1 / 0.40 = 2.5` normalized error, while an error of **1°C** on temperature is only `1 / 7.49 ≈ 0.13`.

Also, rain is often **zero-inflated** and more intermittent than temperature, which makes it harder to model smoothly.


---
## 3. Build the Agent Class

### Exercise 2.1 — Understand the Agent contract

The platform calls your agent like this:

```python
agent = Agent()                     # loads model in __init__
prediction = agent.predict(X_test)  # X_test: (20, 24, 8) → returns (3,)
```

The baseline agent (last known Paris values):

```python
class Agent:
    def predict(self, X_test):
        paris = X_test[16]  # (24, 8)
        return np.array([paris[-1, 0], paris[-1, 2], paris[-1, 1]])
        # [last temperature, last wind_speed, last rain]
```

Note the output order: `[temperature, wind_speed, rain]` — but in the input features, rain is index 1 and wind_speed is index 2.

### Exercise 2.2 — Bridge TP1 → TP2: from DataFrame to 3D array

In TP1, you built features from a DataFrame (pivot table with columns like `Paris_temperature`, `London_rain`, etc.).

On the platform, you get a **3D numpy array** `(20, 24, 8)`. You need to extract the same features.

**Important:** `pivot_table` sorts columns **alphabetically** by feature name. So the column order is: `clouds_Amsterdam, clouds_Barcelona, ..., humidity_Amsterdam, ..., rain_Amsterdam, ..., snow_Amsterdam, ..., temperature_Amsterdam, ..., wind_direction_Amsterdam, ..., wind_speed_Amsterdam, ...`. Your feature extraction loop must follow this same alphabetical order.

Here's the mapping:

```python
PARIS_IDX = 16

# TP1: df['temperature_Paris']  →  TP2: X_test[16, :, 0]
# TP1: df['rain_London']        →  TP2: X_test[11, :, 1]
# TP1: df['temperature_Paris_lag3']  →  TP2: X_test[16, -3, 0]
# TP1: df['wind_speed_Munich']  →  TP2: X_test[15, :, 2]
```

Write a function `extract_features(X_test)` that extracts the same features you used in TP1 from the 3D array and returns a 1D numpy array.

*Hint:* Loop over feature indices in **alphabetical order** (clouds, humidity, rain, snow, temperature, wind_direction, wind_speed), then for each feature loop over the 20 cities. Add Paris lag features and time features (sin_hour, cos_hour — use 0 as placeholder).

In [ ]:
CITIES = [
    "Amsterdam", "Barcelona", "Birmingham", "Brussels", "Copenhagen",
    "Dortmund", "Dublin", "Düsseldorf", "Essen", "Frankfurt am Main",
    "Köln", "London", "Manchester", "Marseille", "Milan", "Munich",
    "Paris", "Rotterdam", "Stuttgart", "Turin",
]
PARIS_IDX = 16

# Feature indices in X_test axis 2
TEMP_IDX = 0
RAIN_IDX = 1
WIND_IDX = 2
WIND_DIR_IDX = 3
HUMIDITY_IDX = 4
CLOUDS_IDX = 5
VISIBILITY_IDX = 6  # not available in our CSV
SNOW_IDX = 7

# Must match the alphabetical order used by pivot_table in TP1
FEATURE_INDICES = [
    CLOUDS_IDX,
    HUMIDITY_IDX,
    RAIN_IDX,
    SNOW_IDX,
    TEMP_IDX,
    WIND_DIR_IDX,
    WIND_IDX,
]

def extract_features(X_test):
    """Extract the same features as in TP1 from one (20, 24, 8) window."""
    features = []

    # Last observed hour: all 20 cities × 7 weather features
    for feat_idx in FEATURE_INDICES:
        for city_idx in range(20):
            features.append(float(X_test[city_idx, -1, feat_idx]))

    # Paris lag features: T-1 ... T-6
    for lag in range(1, 7):
        features.append(float(X_test[PARIS_IDX, -lag, TEMP_IDX]))
        features.append(float(X_test[PARIS_IDX, -lag, WIND_IDX]))
        features.append(float(X_test[PARIS_IDX, -lag, RAIN_IDX]))

    return np.array(features, dtype=float)


### Exercise 2.3 — Write the full Agent class

Combine `extract_features` and your 3 models into a complete Agent class.

*Hint:* Load `model_temperature.pkl`, `model_wind_speed.pkl`, `model_rain.pkl` in `__init__`, extract features and predict each target in `predict()`.

In [ ]:
class Agent:
    def __init__(self):
        self.model_temperature = joblib.load("model_temperature.pkl")
        self.model_wind_speed = joblib.load("model_wind_speed.pkl")
        self.model_rain = joblib.load("model_rain.pkl")

    def predict(self, X_test):
        x = extract_features(X_test).reshape(1, -1)

        temperature = self.model_temperature.predict(x)[0]
        wind_speed = self.model_wind_speed.predict(x)[0]

        # rain model was trained only on current Paris rain
        rain_feature = np.array([[x[0, 56]]], dtype=float)  # rain_Paris in the flattened vector
        rain = self.model_rain.predict(rain_feature)[0]

        return np.array([temperature, wind_speed, rain], dtype=float)


---
## 4. Test Agent Locally

Before submitting, let's test the agent with real data to make sure it works correctly.

### Exercise 3.1 — Build a real `(20, 24, 8)` window from the CSV

Load `weather_paris_20cities.csv` and build a test window:
1. Pick 24 consecutive hours (e.g., 2025-06-15 00:00 to 2025-06-15 23:00)
2. For each hour, get the 20 cities (sorted alphabetically)
3. Extract the 8 features: temperature, rain, wind_speed, wind_direction, humidity, clouds, visibility (use 0 if missing), snow
4. Result: array of shape `(20, 24, 8)`

Also get the ground truth: Paris weather at T+6h (6 hours after the last hour in the window).

*Hint:*
```python
feature_cols = ['temperature', 'rain', 'wind_speed', 'wind_direction', 
                'humidity', 'clouds', 'visibility', 'snow']
# Note: visibility is not in our CSV — use 0 as placeholder (index 6)
```

In [ ]:
feature_cols = [
    "temperature", "rain", "wind_speed", "wind_direction",
    "humidity", "clouds", "visibility", "snow"
]

df = pd.read_csv("weather_paris_20cities.csv", parse_dates=["timestamp"])
df = df[(df["city_name"] != "Barcelona") | (df["country_code"] == "ES")].copy()
df = df.sort_values(["timestamp", "city_name"]).copy()

weather_cols = ["temperature", "rain", "wind_speed", "wind_direction",
                "humidity", "clouds", "snow"]

for col in weather_cols:
    df[col] = (
        df.groupby("city_name")[col]
          .transform(lambda s: s.interpolate(limit_direction="both").ffill().bfill())
    )

# Clip same columns as in TP1
for col in ["temperature", "humidity", "wind_speed"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df[col] = df[col].clip(lower, upper)

# Add missing competition feature
df["visibility"] = 0.0

# Pick a real 24-hour window
start_ts = pd.Timestamp("2025-06-15 00:00:00+00:00")
end_ts = start_ts + pd.Timedelta(hours=23)

window = df[(df["timestamp"] >= start_ts) & (df["timestamp"] <= end_ts)].copy()
window["city_name"] = pd.Categorical(window["city_name"], categories=CITIES, ordered=True)
window = window.sort_values(["city_name", "timestamp"])

# Build (20, 24, 8)
X_test = np.zeros((20, 24, 8), dtype=float)

for city_idx, city in enumerate(CITIES):
    city_window = window[window["city_name"] == city].sort_values("timestamp")
    X_test[city_idx, :, :] = city_window[feature_cols].to_numpy()

# Ground truth Paris at T+6h from the end of the window
target_ts = end_ts + pd.Timedelta(hours=6)
paris_truth = df[(df["timestamp"] == target_ts) & (df["city_name"] == "Paris")].iloc[0]
ground_truth = np.array([
    paris_truth["temperature"],
    paris_truth["wind_speed"],
    paris_truth["rain"],
], dtype=float)

print("X_test shape:", X_test.shape)
print("Ground truth:", ground_truth)


### Exercise 3.2 — Run the agent and check output

Instantiate your `Agent` and call `predict(X_test)`. Verify:
- Output shape is `(3,)`
- Values are reasonable (temperature in [-10, 45], wind_speed in [0, 30], rain in [0, 20])

*Hint:* `agent = Agent()`, `pred = agent.predict(X_test)`, `print(pred.shape, pred)`

In [ ]:
agent = Agent()
prediction = agent.predict(X_test)

print("Prediction shape:", prediction.shape)
print("Prediction:", prediction)


### Exercise 3.3 — Reproduce platform evaluation

Compare your prediction with the actual Paris values at T+6h. Compute the Negated Normalized MAE — this is what you'll see on the leaderboard.

```python
std = np.array([7.49, 5.05, 0.40])  # [temperature, wind_speed, rain]
score = -np.mean(np.abs(prediction - ground_truth) / std)
```

Try with multiple windows to get a sense of your model's average performance.

*Hint:* Loop over several start times, build window + ground truth, predict, compute score, then average.

In [ ]:
std = np.array([7.49, 5.05, 0.40])
score = -np.mean(np.abs(prediction - ground_truth) / std)

print("Ground truth:", ground_truth)
print("Prediction:", prediction)
print("Negated Normalized MAE:", score)

# Estimate average local score on multiple windows
scores = []
candidate_starts = pd.date_range("2025-03-01", "2025-12-20", freq="14D", tz="UTC")

for start_ts in candidate_starts:
    end_ts = start_ts + pd.Timedelta(hours=23)
    target_ts = end_ts + pd.Timedelta(hours=6)

    window = df[(df["timestamp"] >= start_ts) & (df["timestamp"] <= end_ts)].copy()
    if len(window) != 20 * 24:
        continue

    truth_rows = df[(df["timestamp"] == target_ts) & (df["city_name"] == "Paris")]
    if len(truth_rows) != 1:
        continue

    window["city_name"] = pd.Categorical(window["city_name"], categories=CITIES, ordered=True)
    window = window.sort_values(["city_name", "timestamp"])

    X_tmp = np.zeros((20, 24, 8), dtype=float)
    ok = True
    for city_idx, city in enumerate(CITIES):
        city_window = window[window["city_name"] == city].sort_values("timestamp")
        if len(city_window) != 24:
            ok = False
            break
        X_tmp[city_idx, :, :] = city_window[feature_cols].to_numpy()

    if not ok:
        continue

    truth = truth_rows.iloc[0]
    y_true = np.array([truth["temperature"], truth["wind_speed"], truth["rain"]], dtype=float)
    y_pred = agent.predict(X_tmp)
    s = -np.mean(np.abs(y_pred - y_true) / std)
    scores.append(s)

print("Average local score over sampled windows:", np.mean(scores))
print("Number of evaluated windows:", len(scores))


---
## 5. Submit via Web Interface

### Exercise 4.1 — Save your `agent.py` file

Your `agent.py` file must contain a class `Agent` with:
- `__init__(self)` — loads `model_temperature.pkl`, `model_wind_speed.pkl`, `model_rain.pkl`
- `predict(self, X_test)` — takes `(20, 24, 8)`, returns `(3,)`

A template is provided in `session3/agent.py`. Adapt it to match the features your model expects.

Save your final version:

In [ ]:
from pathlib import Path

agent_code = '''
import numpy as np
import joblib

PARIS_IDX = 16

TEMP_IDX = 0
RAIN_IDX = 1
WIND_IDX = 2
WIND_DIR_IDX = 3
HUMIDITY_IDX = 4
CLOUDS_IDX = 5
VISIBILITY_IDX = 6
SNOW_IDX = 7

FEATURE_INDICES = [
    CLOUDS_IDX,
    HUMIDITY_IDX,
    RAIN_IDX,
    SNOW_IDX,
    TEMP_IDX,
    WIND_DIR_IDX,
    WIND_IDX,
]

def extract_features(X_test):
    features = []
    for feat_idx in FEATURE_INDICES:
        for city_idx in range(20):
            features.append(float(X_test[city_idx, -1, feat_idx]))
    for lag in range(1, 7):
        features.append(float(X_test[PARIS_IDX, -lag, TEMP_IDX]))
        features.append(float(X_test[PARIS_IDX, -lag, WIND_IDX]))
        features.append(float(X_test[PARIS_IDX, -lag, RAIN_IDX]))
    return np.array(features, dtype=float)

class Agent:
    def __init__(self):
        self.model_temperature = joblib.load("model_temperature.pkl")
        self.model_wind_speed = joblib.load("model_wind_speed.pkl")
        self.model_rain = joblib.load("model_rain.pkl")

    def predict(self, X_test):
        x = extract_features(X_test).reshape(1, -1)
        temperature = self.model_temperature.predict(x)[0]
        wind_speed = self.model_wind_speed.predict(x)[0]
        rain = self.model_rain.predict(np.array([[x[0, 56]]], dtype=float))[0]
        return np.array([temperature, wind_speed, rain], dtype=float)
'''

Path("agent.py").write_text(agent_code)
print("Saved agent.py")


### Exercise 4.2 — Submit to the competition

1. Go to [ml-arena.com/viewcompetition/26](https://ml-arena.com/viewcompetition/26)
2. Click **"Submit Agent"**
3. Upload your files:
   - `agent.py` — your Agent class
   - `model_temperature.pkl` — temperature model
   - `model_wind_speed.pkl` — wind speed model
   - `model_rain.pkl` — rain model
4. Wait for the status to show **"Running"** then **"Completed"**
5. Check your score on the leaderboard

If the status shows **"Failed"**, check the error message — common issues:
- Wrong output shape (must be `(3,)`)
- Import errors (only `numpy`, `pandas`, `scikit-learn`, `joblib` are available)
- Feature mismatch between training and prediction

---
## 6. Ideas for Improvement

You have **~1 month by team** to iterate and climb the leaderboard. Here are ideas to explore:

### Rain is the hardest target
Rain has `std = 0.40 mm`, so being off by just 1mm costs `1/0.40 = 2.5` in normalized error. Temperature has `std = 7.49`, so 1°C off only costs `0.13`. Focus effort on rain prediction.

### Use neighboring cities as predictors
Weather systems move — typically west → east in Europe. London/Dublin weather a few hours ago can predict what's coming to Paris. Look at which cities are most correlated with Paris at different lags.

### Wind direction as a signal
Westerly winds (from the Atlantic) bring rain and mild temperatures. Easterly winds bring continental weather (cold/dry in winter, hot in summer). Wind direction from upstream cities is especially informative.

### Seasonal adaptation
The 30-day rolling leaderboard rewards models that adapt to the current season. A model trained on full-year data might underperform one tuned to the current month's patterns. Consider retraining or using seasonal features.

### Try different algorithms
- **Boosint/RandomForest**
- **Neural networks** — can capture complex temporal patterns (but need more data/tuning)
- **Ensemble of models** — average predictions from multiple approaches

### More feature ideas
- Rolling statistics (mean/std over last 6/12/24h)
- Differences (temperature change in last 3h)
- Cross-city features (temperature gradient west → east)
- Pressure proxy from humidity + clouds patterns

---
## 7. Summary

### What you did

| Step | What |
|------|------|
| **Understand** | Input `(20, 24, 8)` → output `(3,)`, scoring metric |
| **Build Agent** | Bridged TP1 features to 3D array extraction |
| **Test locally** | Built real windows, reproduced platform scoring |
| **Submit** | Uploaded to ml-arena.com |

### Deadlines

| Date | What |
|------|------|
| **Now** | First submission — make sure it runs! |
| **~April 10, 2026** | Presentation — explain your approach |
| **May 15, 2026** | 30-day rolling leaderboard assessed for final grade |

Good luck — and may the best model win!